# slv_items_pedido
| Campo | Valor |
|---|---|
| Proyecto | Andina Market · Reto técnico de Ingeniería de Datos |
| Capa | Silver (nivel 2) · entidad Ítems de pedido (detalle) |
| Propósito | bronze.order_items_raw → silver.items_pedido (SCD1, incluye los DELETE). monto_item   : cantidad × precio_unitario − descuento. Es la base de las ventas en gold (no el total de cabecera, que en algunos pedidos legacy no cuadra). |
| Reglas | reglas_calidad.REGLAS["items_pedido"] |
| Autoría | Richard Huamán · con asistencia de IA (Claude) |
| Versión | 2.0 · 2026-10-12 |

## 1. Librerías y módulos comunes
- `dlt`: API de los pipelines declarativos (vistas, tablas, AUTO CDC, expectations).
- `comun` / `comun_gold` y `reglas_calidad`: utilidades y reglas compartidas por todas las entidades (archivos `.py` de la misma carpeta; el pipeline los encuentra gracias a `root_path`).

In [ ]:
import dlt
from pyspark.sql import functions as F

from comun import TECNICAS, con_secuencia, leer_bronze, parametros_cdc
from reglas_calidad import reglas

# AUTO CDC: nombre actual de la API; apply_changes es el nombre anterior.
aplicar_cdc = getattr(dlt, "create_auto_cdc_flow", None) or dlt.apply_changes

## 2. Vista de limpieza `limpio_items_pedido`
Lee los cambios de bronze **como stream** (solo lo nuevo en cada corrida) y:
1. agrega el **orden de los cambios** (`_secuencia`: fecha efectiva, `_lsn`, `_seqval`);
2. **renombra** a snake_case en español y **normaliza** valores;
3. aplica las **reglas de calidad** en tres niveles: `fallar` (detiene el pipeline), `descartar` (no entra; la recoge la cuarentena) y `advertir` (entra y queda la métrica).

In [ ]:
@dlt.view(name="limpio_items_pedido", comment="Ítems de pedido de bronze renombrados y validados.")
@dlt.expect_all_or_fail(reglas("items_pedido", "fallar"))      # rompe el modelo → detiene el pipeline
@dlt.expect_all_or_drop(reglas("items_pedido", "descartar"))   # inválido → no entra (va a cuarentena)
@dlt.expect_all(reglas("items_pedido", "advertir"))            # mejorable → entra y queda la métrica
def limpio_items_pedido():
    df = con_secuencia(leer_bronze("order_items_raw"))
    return df.select(
        F.col("OrderItemID").alias("id_item"),
        F.col("OrderID").alias("id_pedido"),
        F.col("ProductID").alias("id_producto"),
        F.col("Quantity").alias("cantidad"),
        F.col("UnitPrice").alias("precio_unitario"),
        F.col("DiscountAmount").alias("descuento"),
        (F.col("Quantity") * F.col("UnitPrice") - F.col("DiscountAmount")).cast("decimal(14,2)").alias("monto_item"),
        F.col("CreatedAt").alias("creado_en"),
        F.col("UpdatedAt").alias("actualizado_en"),
        *TECNICAS)

## 3. Tabla(s) silver con AUTO CDC: `items_pedido`
AUTO CDC aplica los cambios **en su orden real** (`_secuencia`), se queda con el último por clave (resuelve duplicados y llegadas fuera de orden) y convierte los `D` de la fuente en **borrados**.
- **SCD1** → estado actual (una fila por clave).

In [ ]:
dlt.create_streaming_table(name="items_pedido", comment="Ítems de pedido: estado actual (SCD1) desde CDC de Azure SQL, incluidos los borrados.")
aplicar_cdc(target="items_pedido", source="limpio_items_pedido", **parametros_cdc("id_item"))
# Lo que no cumple una regla de descarte lo recoge slv_cuarentena.py.